In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from xgboost import XGBRegressor



In [ ]:
df = pd.read_csv(r"C:\Users\Khan\OneDrive\Desktop\Intern_Performance_prediction\intern_dataset.csv")

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
sns.displot(df['Attendance'], kde=True)
plt.show()

In [ ]:
sns.displot(df['Completion_Time'], kde=True)
plt.show()

In [ ]:
corr = df.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.show()

In [ ]:
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.histplot(df['Performance_Score'], kde=True, color='teal')
plt.title('Performance Score Distribution')

In [ ]:
plt.subplot(1, 2, 2)
sns.boxplot(y=df['Performance_Score'], color='lightcoral')
plt.title('Performance Score Spread & Outliers')
plt.show()

In [ ]:
df = df.drop(columns=["Attendance","Intern_ID"])

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
cols_to_fix = ['Completion_Time', 'Feedback_Rating']
data_subset = df[cols_to_fix]
imputer = IterativeImputer(max_iter=10, random_state=0)
imputed_data = imputer.fit_transform(data_subset)

In [ ]:
df.isnull().sum()

In [ ]:
df[cols_to_fix] = imputer.fit_transform(df[cols_to_fix])

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df["Efficiency"] = df["Feedback_Rating"] / df["Completion_Time"]

In [ ]:
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
sns.histplot(df['Efficiency'], kde=True)
plt.title("Distribution of the New Efficiency Metric")
plt.show()

In [ ]:
X = df[["Completion_Time", "Feedback_Rating", "Efficiency"]]
y = df["Performance_Score"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

In [ ]:
gb_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train, y_train)

In [ ]:
xgb_model = XGBRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5, 
    random_state=42,
    objective='reg:squarederror'
)

xgb_model.fit(X_train, y_train)

# 4. Predict and Evaluate
xgb_preds = xgb_model.predict(X_test)

print(f"XGBoost R2: {r2_score(y_test, xgb_preds):.3f}")
print(f"XGBoost MAE: {mean_absolute_error(y_test, xgb_preds):.3f}")
print(f"XGBoost MSE: {mean_squared_error(y_test, xgb_preds):.3f}")

In [ ]:
rf_preds = rf_model.predict(X_test)
gb_preds = gb_model.predict(X_test)

print(f"Random Forest R2: {r2_score(y_test, rf_preds):.3f}")
print(f"Random Forest MSE: {mean_squared_error(y_test, rf_preds):.3f}")
print(f"Random Forest MAE: {mean_absolute_error(y_test, rf_preds):.3f}")
print(f"Gradient Boosting R2: {r2_score(y_test, gb_preds):.3f}")
print(f"Gradient Boosting MAE: {mean_absolute_error(y_test, gb_preds):.3f}")
print(f"Gradient Boosting MSE: {mean_squared_error(y_test, gb_preds):.3f}")


In [ ]:
importances = gb_model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

print(feature_importance_df)

# Visualize it
plt.barh(feature_importance_df['Feature'], feature_importance_df['Importance'], color='teal')
plt.title('Which factor matters most for the Model?')
plt.show()

In [ ]:
# 1. Use your trained model to predict scores for everyone
df['Predicted_Score'] = gb_model.predict(X)

# 2. Define the thresholds based on percentiles
lower_threshold = df['Predicted_Score'].quantile(0.20) # Bottom 20%
upper_threshold = df['Predicted_Score'].quantile(0.80) # Top 20%

# 3. Create the categorization function
def categorize(score):
    if score >= upper_threshold:
        return 'Excel'
    elif score <= lower_threshold:
        return 'Struggle'
    else:
        return 'Average'

# 4. Apply categorization to a new column
df['Performance_Category'] = df['Predicted_Score'].apply(categorize)

# 5. Filter for your "Action Lists"
excel_interns = df[df['Performance_Category'] == 'Excel']
struggle_interns = df[df['Performance_Category'] == 'Struggle']

print(f"Number of interns likely to Excel: {len(excel_interns)}")
print(f"Number of interns likely to Struggle: {len(struggle_interns)}")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
sns.boxplot(x='Performance_Category', y='Efficiency', data=df, palette='viridis')
plt.title('The Efficiency Gap: Excel vs. Struggle')
plt.show()